# 04_Gold_Aggregations_Optimization
This notebook reads the curated Silver tables, builds business‑ready Gold tables, and applies Delta Lake performance optimizations (OPTIMIZE, Z‑ORDER, COALESCE).
All tables are stored in the `ecommerce_catalog.gold` schema.


In [ ]:
from pyspark.sql import functions as F, Window

CATALOG = "ecommerce_catalog"

# Load Silver tables
orders = spark.table(f"{CATALOG}.silver.orders")

# ------------------------------------------------------------------
#  1️⃣ Daily Sales aggregation
# ------------------------------------------------------------------
daily_sales = (orders
    .filter(F.col("status") == "COMPLETED")
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("total_amount").alias("total_sales"),
        F.avg("total_amount").alias("average_order_value"),
        F.max("total_amount").alias("highest_order_value")
    )
)

# Write Gold daily_sales (coalesce to 2 files)
(daily_sales.coalesce(2).write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.daily_sales"))

# ------------------------------------------------------------------
#  2️⃣ Customer Sales aggregation
# ------------------------------------------------------------------
customer_sales = (orders
    .filter(F.col("status") == "COMPLETED")
    .groupBy("customer_id", "customer_name", "city")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("total_amount").alias("total_spent"),
        F.avg("total_amount").alias("average_order_value")
    ))

# Write Gold customer_sales (coalesce to 2 files)
(customer_sales.coalesce(2).write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.customer_sales"))

# ------------------------------------------------------------------
#  3️⃣ Product Sales aggregation
# ------------------------------------------------------------------
product_sales = (orders
    .filter(F.col("status") == "COMPLETED")
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("total_amount").alias("revenue"),
        F.avg("price").alias("average_price")
    ))

# Write Gold product_sales (coalesce to 2 files)
(product_sales.coalesce(2).write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.product_sales"))

# ------------------------------------------------------------------
#  4️⃣ Top Customers (window function)
# ------------------------------------------------------------------
window_spec = Window.orderBy(F.col("total_spent").desc())
top_customers = (customer_sales
    .withColumn("customer_rank", F.dense_rank().over(window_spec))
    .filter(F.col("customer_rank") <= 10))

# Write Gold top_customers (coalesce to 1 file)
(top_customers.coalesce(1).write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.gold.top_customers"))

# ------------------------------------------------------------------
#  5️⃣ Performance Optimizations (OPTIMIZE + Z‑ORDER)
# ------------------------------------------------------------------
# Optimize Silver orders table and Z‑ORDER by high‑cardinality columns
spark.sql(f"OPTIMIZE {CATALOG}.silver.orders ZORDER BY (customer_id, order_date)")

# Optimize Gold tables
spark.sql(f"OPTIMIZE {CATALOG}.gold.daily_sales ZORDER BY (order_date)")
spark.sql(f"OPTIMIZE {CATALOG}.gold.customer_sales ZORDER BY (customer_id)")
spark.sql(f"OPTIMIZE {CATALOG}.gold.product_sales ZORDER BY (product_id)")

print("Gold layer built and optimized.")
